# External and formula-backed symmetry candidates

This notebook is dedicated to the v0.16-v0.17 interop surface. It validates fitted `GeneratorFamily`, finite `InvariantMapSpec`, formula-backed `FormulaGeneratorFamily`, and one deliberately failed candidate.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np

from notebooks._tutorial_utils import confidence_card, print_cards, pretty_json
from pdelie import GeneratorFamily, InvariantMapSpec
from pdelie.data import generate_heat_1d_field_batch
from pdelie.reporting import summarize_formula_generator_family, summarize_generator_fit_diagnostics
from pdelie.residuals import HeatResidualEvaluator
from pdelie.symmetry import FormulaGeneratorFamily, fit_translation_generator, validate_symmetry_candidate

DOMAIN_LENGTH = 2.0 * np.pi
CONFIG = {"fit_epsilon": 1e-4, "finite_shift": DOMAIN_LENGTH / 8.0}
CONFIG


## 1. Build one field and four candidate styles


In [ ]:
field = generate_heat_1d_field_batch(batch_size=3, num_times=17, num_points=64, seed=670)
evaluator = HeatResidualEvaluator()
fitted = fit_translation_generator(field, evaluator, epsilon=CONFIG["fit_epsilon"])
spec = InvariantMapSpec(
    generator_metadata=fitted.to_dict(),
    construction_method="uniform_translation",
    parameters={"axis": "x", "shift": CONFIG["finite_shift"]},
    domain_validity="global",
    inverse_available=True,
    diagnostics={},
)
formula = FormulaGeneratorFamily(
    formula_generators=[
        {
            "name": "formula_translation",
            "components": {
                "tau": {"node": "const", "value": 0.0},
                "xi": {"node": "const", "value": 1.0},
                "phi": {"node": "const", "value": 0.0},
            },
        }
    ],
    finite_transform_spec=spec.to_dict(),
)
failed = GeneratorFamily(
    parameterization=fitted.parameterization,
    coefficients=np.asarray([[0.0, 0.0, 1.0, 0.0]], dtype=float),
    basis_spec=fitted.basis_spec,
    normalization=fitted.normalization,
    diagnostics={},
)
print(pretty_json(summarize_formula_generator_family(formula), max_chars=2500))


## 2. Validate candidates and compare conclusions


In [ ]:
candidates = {
    "fitted_generator": fitted,
    "invariant_map_payload": spec.to_dict(),
    "formula_generator": formula,
    "failed_wrong_span": failed,
}
reports = {
    name: validate_symmetry_candidate(
        field,
        candidate,
        residual_evaluator=evaluator,
        source_candidate_id=name,
    )
    for name, candidate in candidates.items()
}
fit_summary = summarize_generator_fit_diagnostics(fitted)
cards = [
    confidence_card(
        label=name,
        fit=fit_summary if name == "fitted_generator" else None,
        validation=report,
    )
    for name, report in reports.items()
]
print_cards(cards)


## 3. A validation report is a configured empirical report

`validated` is not a theorem. It means configured checks passed under this field, evaluator, epsilons, reference, and thresholds.


In [ ]:
print(pretty_json({
    name: {
        "candidate_kind": report["candidate_kind"],
        "conclusion": report["conclusion"],
        "configured_checks": report["configured_validation_checks"],
    }
    for name, report in reports.items()
}, max_chars=5000))


## Takeaway

External methods can slot into PDELie by producing structured candidates. PDELie validates those outputs without becoming a detector-training framework.
